In [1]:
## Doublet removal through Doublet Detection
## Script was repeated for each file to removal doublets.
import numpy as np
import doubletdetection
import scanpy as sc
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
adata = sc.read_h5ad(
    "STH.h5ad"
)

In [3]:
sc.pp.filter_cells(adata, min_counts=500)  
sc.pp.filter_cells(adata, min_genes=200)    
sc.pp.filter_genes(adata, min_cells=5)

In [ ]:
clf = doubletdetection.BoostClassifier(
    clustering_algorithm="leiden", 
    standard_scaling=True,
    pseudocount=0.1,
    n_jobs=-1,
)
doublets = clf.fit(adata.X).predict(p_thresh=1e-8, voter_thresh=0.7)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/doubletdetection/doubletdetection.py:106: UserWarning: Leiden clustering is experimental and results have not been validated.
  warnings.warn("Leiden clustering is experimental and results have not been validated.")


  0%|          | 0/10 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Library/Frameworks/Python.framework/Versions/3.13/lib/pyth

In [ ]:
doublet_score = clf.doublet_score()

In [ ]:
adata.obs["doublet"] = doublets
adata.obs["doublet_score"] = doublet_score

In [ ]:
doublets = doubletdetection.plot.convergence(clf, save='convergence_test.pdf', show=True, p_thresh=1e-7, voter_thresh=0.8)


In [ ]:
adata

In [ ]:
sc.pl.violin(adata, "doublet_score")


In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["doublet", "doublet_score"])


In [ ]:
adata_singlets = adata[adata.obs['doublet'] == 0].copy()
## Singlets are 0, 1 is doublets


In [ ]:
adata_singlets

In [ ]:
adata_singlets.obs.drop(columns=['doublet', 'doublet_score'], inplace=True)
adata_singlets.write_h5ad("STH_singlet.h5ad", compression = 'gzip')